In [11]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client['TiendaBigData'] 
coleccion = db['SantaIsabelLimpiezaBano']  #Cambiado para productos de limpieza Jumbo

print("Conexión exitosa a MongoDB")
print(f"Base de datos: TiendaBigData")
print(f"Colección: SantaIsabelLimpiezaBano")

Conexión exitosa a MongoDB
Base de datos: TiendaBigData
Colección: SantaIsabelLimpiezaBano


In [14]:
import os
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

os.system("pkill -9 chrome")
print("🧹 Limpieza OK")

options = Options()
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1366,768")

driver = webdriver.Chrome(options=options)
datos_finales = []

try:
    print("🌐 Santa Isabel - Limpieza Baño")
    driver.get("https://www.santaisabel.cl/limpieza/bano")
    time.sleep(12)
    
    bloques = driver.find_elements(By.CSS_SELECTOR, 
        "div[class*='product'], div[class*='Product'], div[class*='item'], div[class*='tile']"
    )
    
    # 🎯 FILTROS MEJORADOS para nombres LIMPIOS
    palabras_limpieza = [
        'limpia', 'deterg', 'jabon', 'cloro', 'desinf', 'pato', 'ajax', 
        'sapolio', 'squat', 'cif', 'glade', 'fabuloso', 'inodoro', 
        'baño', 'vidrio', 'pisos', 'multi', 'superf'
    ]
    
    nombres_validos = set()
    exitosos = 0
    
    for bloque in bloques:
        try:
            # 🏷️ NOMBRE: Solo líneas con palabras de limpieza
            texto_bloque = bloque.text
            lineas = [l.strip() for l in texto_bloque.split('\n') if len(l.strip()) > 5]
            
            nombre_candidato = None
            for linea in lineas:
                if any(palabra in linea.lower() for palabra in palabras_limpieza):
                    nombre_candidato = linea
                    break
            
            if not nombre_candidato:
                continue
            
            # 💰 PRECIO: Primer $ válido
            precio_elems = bloque.find_elements(By.XPATH, ".//*[contains(text(), '$')]")
            precio = 0.0
            
            for p_elem in precio_elems:
                p_texto = p_elem.text.strip()
                # Ignorar precios con "x un", "x kg" (son PPU)
                if any(unit in p_texto.lower() for unit in ['x un', 'x kg', 'x lt']):
                    continue
                
                p_limpio = ''.join(c for c in p_texto if c.isdigit())
                if len(p_limpio) >= 3:
                    precio = float(p_limpio)
                    break
            
            if precio > 0 and nombre_candidato not in nombres_validos:
                nombres_validos.add(nombre_candidato)
                
                datos_finales.append({
                    "nombre": nombre_candidato,
                    "precio": round(precio),  # Redondear
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": "SantaIsabel_Limpieza_Bano",
                    "categoria": "Limpieza Baño",
                    "tienda": "Santa Isabel",
                    "precio_formateado": f"${precio:,.0f}"
                })
                
                exitosos += 1
                print(f"✅ {exitosos:2d} | {nombre_candidato[:55]:<55} | ${precio:>8,.0f}")
                
                if exitosos >= 25:
                    break
                    
        except:
            continue
    
    print(f"\n🎉 {exitosos} productos de LIMPIEZA capturados")

except Exception as e:
    print(f"❌ {e}")

finally:
    driver.quit()

# 💾 MONGODB ORDENADO
if datos_finales:
    client = MongoClient('mongodb://mongodb:27017/')
    db = client['TiendaBigData']
    coleccion = db['SantaIsabelLimpiezaBano']
    
    # Limpiar duplicados y ordenar por precio
    únicos_ordenados = []
    vistos = set()
    for d in sorted(datos_finales, key=lambda x: x['precio']):
        if d['nombre'] not in vistos:
            únicos_ordenados.append(d)
            vistos.add(d['nombre'])
    
    coleccion.delete_many({})  # Limpiar colección anterior
    coleccion.insert_many(únicos_ordenados)
    
    print(f"\n💾 {len(únicos_ordenados)} productos LIMPIOS guardados")
    print("📊 ESTADÍSTICAS CANASTA BÁSICA:")
    print(f"   Promedio: ${sum(p['precio'] for p in únicos_ordenados)/len(únicos_ordenados):,.0f}")
    print(f"   Mínimo:   ${min(p['precio'] for p in únicos_ordenados):,.0f}")
    print(f"   Máximo:   ${max(p['precio'] for p in únicos_ordenados):,.0f}")
    
    print("\n🏆 TOP 10 MÁS BARATOS (Canasta Básica):")
    for i, p in enumerate(únicos_ordenados[:10], 1):
        print(f"   {i:2d}. ${p['precio']:>7,.0f} | {p['nombre']}")

🧹 Limpieza OK
🌐 Santa Isabel - Limpieza Baño
✅  1 | Pato Purific                                            | $   3,450
✅  2 | Cloro Clorinda Tradicional 1 kg                         | $   1,350
✅  3 | Clorox                                                  | $   2,130
✅  4 | Cloro Gel Clorinda Lavanda 900 ml                       | $   1,890
✅  5 | Cloro Gel Clorinda Menta 900 ml                         | $   1,890
✅  6 | Cloro Clorinda Tradicional 2 kg                         | $   2,230
✅  7 | Cloro Gel Cif Normal 800 ml                             | $17,502,190
✅  8 | Toallas Húmedas Virutex Desinfectante Easy Clean 90 un. | $36,724,590
✅  9 | Cloro Máxima Tradicional 1.9 kg                         | $   1,350
✅ 10 | Cloro Gel Clorinda Regular 900 ml                       | $   1,890
✅ 11 | Cloro Gel Home Care Menta 900 ml                        | $15,901,850
✅ 12 | Cloro Gel Clorinda Flores Silvestres 900 ml             | $   1,890
✅ 13 | Cloro Clorinda Tradicional 4 kg           